# 03 — Champion and Baseline Benchmarks

Select champions, audit existing results, then run benchmark trials. Execution parameters are defined below; incompatible cached results require new trials.


In [1]:
# Setup paths and services
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import pandas as pd
from IPython.display import display, HTML

from shared.config import RESULTS_DIR
from benchmark.infra.di import BenchmarkProvider
from benchmark.application.trials.coordinator import BenchmarkCoordinator
from benchmark.application.select_champions import ChampionSelectionService
from benchmark.infra.storage import SQLiteSynthesisReadRepository
from shared.infra.lifecycle import WorkflowScope


CHAMPIONS_PATH = RESULTS_DIR / 'benchmark' / 'champions.json'

# The benchmark protocol is explicit; candidate conditions come from the database.

print('✅ Benchmark evaluation environment initialized.')
primary_target = 1e-8
secondary_target = 1e-2

# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(BenchmarkProvider(project_root=root_dir), context={}, request_context={})
benchmark_coordinator = services.get(BenchmarkCoordinator)
champion_selection = services.get(ChampionSelectionService)
synthesis_reader = services.get(SQLiteSynthesisReadRepository)



✅ Benchmark evaluation environment initialized.
Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/benchmark/2026-11-21_19-29-36_UTC-42dd2160/diagnostics.log


## 1. Select champions


In [2]:
summary, total_completed = synthesis_reader.get_experiment_balance()

# Refresh the exported champion snapshot from the database currently loaded in this project.
champions_dict_result = champion_selection.export_champions(CHAMPIONS_PATH)
if champions_dict_result.is_success:
    champions_dict, df_champions = champions_dict_result.value
    print(f'✅ Exported {len(df_champions)} champions to {CHAMPIONS_PATH}')
    
    if summary.empty:
        raise RuntimeError('No completed experiments found in database. Please run Notebook 02 first.')
    
    print(f'== Total Completed Experiments: {total_completed} ==')
    print(summary.to_string(index=False))
else:
    champions_dict = None
    df_champions = None
    print(f"Error: {champions_dict_result.error.message}")

    # The snapshot above is refreshed before evaluation, so it always matches the active DB.


✅ Exported 1520 champions to /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/benchmark/champions.json
== Total Completed Experiments: 1542 ==
 problem_id  dim  noise_std     mode prompt_strategy                               llm_name  completed_count
          1    2       0.00 explicit        baseline                  gwdg.qwen3-coder-next                1
          1    2       0.00 explicit        baseline qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit        baseline qwen2.5-coder-32b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit          guided                  gwdg.qwen3-coder-next                1
          1    2       0.00 explicit          guided qwen2.5-coder-14b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit          guided qwen2.5-coder-32b-instruct-q4_k_m.gguf                1
          1    2       0.00 explicit        thinking                  gwdg.qwen3-coder-next  

## 2. Audit benchmark readiness


In [3]:
# ── Embedded HTML Dashboard Helper (Notebook-Local UI) ─────────────────────────
def render_html_dashboard(
    df_audit: pd.DataFrame,
    title: str = "Benchmark Evaluation Pre-Flight Audit",
    subtitle: str = "Real-time status of empirical evaluation runs",
    group_column: str = "model",
) -> str:
    """Render responsive HTML pre-flight dashboard for Jupyter display."""
    if df_audit.empty:
        return "<div>No audit data available.</div>"

    grp_col = (
        group_column
        if group_column in df_audit.columns
        else ("solver" if "solver" in df_audit.columns else df_audit.columns[0])
    )


    summary_rows = []
    for name, grp in df_audit.groupby(grp_col):
        clean_name = str(name) if grp_col == "model" else str(name).upper()
        total = len(grp)
        completed = len(grp[grp["status"] == "COMPLETED"])
        pending = len(grp[grp["status"] == "PENDING"])
        needs_rerun = len(grp[grp["status"] == "NEEDS_RERUN"])
        missing_code = len(grp[grp["status"] == "MISSING_CODE"])
        to_run_mask = grp["status"].isin(["PENDING", "NEEDS_RERUN"])
        if "is_filtered" in grp.columns:
            to_run_mask = to_run_mask & (~grp["is_filtered"])
        to_run = len(grp[to_run_mask])
        pct = (completed / total * 100) if total > 0 else 0.0
        summary_rows.append({
            "Group": clean_name,
            "Total Tasks": total,
            "Completed": completed,
            "Pending": pending,
            "Needs Rerun": needs_rerun,
            "Missing Code": missing_code,
            "Queue to Run": to_run,
            "Progress (%)": pct,
        })

    df_summary = pd.DataFrame(summary_rows)
    total_queue = int(df_summary["Queue to Run"].sum())
    total_completed = int(df_summary["Completed"].sum())
    total_tasks = int(df_summary["Total Tasks"].sum())
    overall_pct = (total_completed / total_tasks * 100) if total_tasks > 0 else 0.0

    html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                🚀 {title}
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">{subtitle}</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            {total_tasks} Target Conditions
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Targets</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{total_tasks}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Across {len(df_summary)} Categories</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{total_completed}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{overall_pct:.1f}% Overall Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{total_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">Pending / Needs Rerun</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Overall Status</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">{overall_pct:.0f}%</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">Completion Rate</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Completion Progress Breakdown
        </div>
"""
    for _, row in df_summary.iterrows():
        grp_name = row["Group"]
        tot = int(row["Total Tasks"])
        comp = int(row["Completed"])
        pend = int(row["Pending"])
        rerun = int(row["Needs Rerun"])
        q = int(row["Queue to Run"])
        pct = float(row["Progress (%)"])
        color = "#10B981" if pct > 75 else ("#3B82F6" if pct > 25 else "#F59E0B")
        rerun_pct = (rerun / tot * 100) if tot > 0 else 0

        html += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>⚙️ <strong style="color: #0F172A;">{grp_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
"""
    html += """
    </div>
</div>
"""
    return html

# 2. Audit champions workload (general overview by model)
df_champ_audit_result = benchmark_coordinator.audit_workload(
    solver_type='champions',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
df_champ_audit = df_champ_audit_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if df_champ_audit is not None:
    print(f"🏆 Champions Workload: {len(df_champ_audit)} conditions ({df_champ_audit["status"].value_counts().to_dict()})")
    display(HTML(render_html_dashboard(df_champ_audit, title="🏆 LLaMEA Champions Evaluation Audit", subtitle="Empirical benchmark evaluation status across LLM-evolved algorithms", group_column="model")))
    
    # 3. Discover unique conditions for classical baselines (general overview by solver)
    df_base_audit_result = benchmark_coordinator.audit_workload(
        solver_type='baselines',
        trials_per_condition=20,
        random_seed=42,
        first_instance_id=2,
        budget=100_000,
        trial_timeout_seconds=60.0,
        observation_trace_points=64,
        force_rerun=False,
        classical_baselines=['cmaes', 'de', 'pso'],
        noise_robustness_enabled=True,
        cross_function_enabled=False,
        cross_function_problem_ids=[1, 8, 11, 15, 21],
    )
    if df_base_audit_result.is_success:
        df_base_audit = df_base_audit_result.value
        print(f"⚙️ Baselines Workload: {len(df_base_audit)} conditions ({df_base_audit["status"].value_counts().to_dict()})")
        display(HTML(render_html_dashboard(df_base_audit, title="⚙️ Classical Baselines Evaluation Audit", subtitle="Empirical benchmark evaluation status across CMA-ES, DE, and PSO", group_column="solver")))
    else:
        df_base_audit = None
        print(f"Error: {df_base_audit_result.error.message}")


🏆 Champions Workload: 1520 conditions ({'COMPLETED': 1458, 'PENDING': 62})


⚙️ Baselines Workload: 237 conditions ({'COMPLETED': 237})


## 3. Benchmark champions


In [4]:
results_champ_df_result = benchmark_coordinator.run_benchmarks(
    solver_type='champions',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
results_champ_df = results_champ_df_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if results_champ_df is not None:
    print(f'✅ LLaMEA Champions evaluation complete: {len(results_champ_df)} conditions processed.')



🚀 STARTING BENCHMARKS FOR CHAMPIONS
   1520 target conditions to evaluate

[1/1520] 🏆 Champion: gwdg.qwen3-coder-next
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0       Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[2/1520] 🏆 Champion: gwdg.qwen3-coder-next
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.05    Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.228). Skipping.

[3/1520] 🏆 Champion: gwdg.qwen3-coder-next
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.1     Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[4/1520] 🏆 Champion: gwdg.qwen3-coder-next
  Mode:        explicit   Strategy:    baseline       Dim:         2D     Noise level: σ=0.2     Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000).

KeyboardInterrupt: 

## 4. Benchmark CMA-ES, DE and PSO


In [ ]:
results_base_df_result = benchmark_coordinator.run_benchmarks(
    solver_type='baselines',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
results_base_df = results_base_df_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if results_base_df is not None:
    print(f'✅ Classical Baselines evaluation complete: {len(results_base_df)} conditions processed.')



🚀 STARTING BENCHMARKS FOR BASELINES
   237 target conditions to evaluate

[1/237] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f1 (Sphere (f1))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[2/237] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f8 (Rosenbrock (f8))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[3/237] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f11 (Discus (f11))
  📦 [CACHED] 20 runs found (Median Δy Error: 0.000). Skipping.

[4/237] ⚙️ Baseline: CMA-ES
  Mode:        —          Strategy:    —              Dim:         2D     Noise level: σ=0       Problem:     f15 (Rastrigin Multi-Modal (f15))
  📦 [CACHED] 20 runs found (Median Δy Error: 1.990). Skipping.

[5/237] ⚙️ Baseline: CMA-E

/Users/nicolaibrahim/Desktop/proj/AAD_LLM/.venv/lib/python3.13/site-packages/cma/s.py:16: UserWarning: Could not import matplotlib.pyplot, therefore ``cma.plot()`` etc. is not available
  _warnings.warn('Could not import matplotlib.pyplot, therefore'


  • ⚡ Trial 1/20 | Instance: 2 | Seed: 44 | f-opt: -92.090 | Best f: 453.243 | Δy Error: 545.333 | Evals: 11,220 | Time: 0.500s
  • ⚡ Trial 2/20 | Instance: 3 | Seed: 45 | f-opt: -87.890 | Best f: -64.580 | Δy Error: 23.310 | Evals: 6,650 | Time: 0.312s
  • ⚡ Trial 3/20 | Instance: 4 | Seed: 46 | f-opt: 320.190 | Best f: 1238.225 | Δy Error: 918.035 | Evals: 11,150 | Time: 0.444s
  • ⚡ Trial 4/20 | Instance: 5 | Seed: 47 | f-opt: -104.240 | Best f: 57.006 | Δy Error: 161.246 | Evals: 6,450 | Time: 0.299s
  • ⚡ Trial 5/20 | Instance: 6 | Seed: 48 | f-opt: -35.700 | Best f: 2408.309 | Δy Error: 2444.009 | Evals: 5,050 | Time: 0.176s
  • ⚡ Trial 6/20 | Instance: 7 | Seed: 49 | f-opt: 20.700 | Best f: 37.698 | Δy Error: 16.998 | Evals: 8,450 | Time: 0.431s
  • ⚡ Trial 7/20 | Instance: 8 | Seed: 50 | f-opt: 38.070 | Best f: 738.818 | Δy Error: 700.748 | Evals: 9,950 | Time: 0.449s
  • ⚡ Trial 8/20 | Instance: 9 | Seed: 51 | f-opt: -45.420 | Best f: 576.462 | Δy Error: 621.882 | Evals: 7,860

/Users/nicolaibrahim/Desktop/proj/AAD_LLM/.venv/lib/python3.13/site-packages/scipy/_lib/_util.py:530: UserWarning: [BUDGET OVERRUN] Problem evaluation called after budget exhausted (100000 >= 100000). Returning last observed fitness.
  return self.f(x, *self.args)


  • ⚡ Trial 2/20 | Instance: 3 | Seed: 45 | f-opt: 115.680 | Best f: 154.146 | Δy Error: 38.466 | Evals: 100,000 | Time: 1.939s
  • ⚡ Trial 3/20 | Instance: 4 | Seed: 46 | f-opt: -3.710 | Best f: 120.603 | Δy Error: 124.313 | Evals: 100,000 | Time: 2.922s
  • ⚡ Trial 4/20 | Instance: 5 | Seed: 47 | f-opt: 132.180 | Best f: 274.116 | Δy Error: 141.936 | Evals: 100,000 | Time: 2.149s
  • ⚡ Trial 5/20 | Instance: 6 | Seed: 48 | f-opt: -584.730 | Best f: -534.973 | Δy Error: 49.757 | Evals: 100,000 | Time: 1.951s
  • ⚡ Trial 6/20 | Instance: 7 | Seed: 49 | f-opt: -36.540 | Best f: 24.622 | Δy Error: 61.162 | Evals: 100,000 | Time: 1.777s
  • ⚡ Trial 7/20 | Instance: 8 | Seed: 50 | f-opt: 1000.000 | Best f: 1142.878 | Δy Error: 142.878 | Evals: 93,900 | Time: 2.666s
  • ⚡ Trial 8/20 | Instance: 9 | Seed: 51 | f-opt: 46.610 | Best f: 131.608 | Δy Error: 84.998 | Evals: 100,000 | Time: 3.774s
  • ⚡ Trial 9/20 | Instance: 10 | Seed: 52 | f-opt: -264.310 | Best f: -198.648 | Δy Error: 65.662 | 

## Optional: noise robustness and cross-function transfer

These separate studies freeze champions. The switches below default to audit-only; enable execution only when intended.


In [ ]:
RUN_NOISE_ROBUSTNESS = False
RUN_CROSS_FUNCTION = False

noise_audit_result = benchmark_coordinator.audit_workload(
    solver_type='noise_robustness',
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
)
noise_audit = noise_audit_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if noise_audit is not None:
    transfer_audit_result = benchmark_coordinator.audit_workload(
        solver_type='cross_function',
        trials_per_condition=20,
        random_seed=42,
        first_instance_id=2,
        budget=100_000,
        trial_timeout_seconds=60.0,
        observation_trace_points=64,
        force_rerun=False,
        classical_baselines=['cmaes', 'de', 'pso'],
        noise_robustness_enabled=True,
        cross_function_enabled=False,
        cross_function_problem_ids=[1, 8, 11, 15, 21],
    )
    if transfer_audit_result.is_success:
        transfer_audit = transfer_audit_result.value
        print(f"Noise robustness: {len(noise_audit)} conditions")
        print(f"Cross-function generalization: {len(transfer_audit)} conditions (opt-in)")
        display(noise_audit)
        display(transfer_audit)
        
        if RUN_NOISE_ROBUSTNESS:
            noise_results_result = benchmark_coordinator.run_benchmarks(
                solver_type='noise_robustness',
                trials_per_condition=20,
                random_seed=42,
                first_instance_id=2,
                budget=100_000,
                trial_timeout_seconds=60.0,
                observation_trace_points=64,
                force_rerun=False,
                classical_baselines=['cmaes', 'de', 'pso'],
                noise_robustness_enabled=True,
                cross_function_enabled=False,
                cross_function_problem_ids=[1, 8, 11, 15, 21],
            )
            if noise_results_result.is_success:
                noise_results = noise_results_result.value
            else:
                noise_results = None
                print(f"Error: {noise_results_result.error.message}")
        if RUN_CROSS_FUNCTION:
            if transfer_audit.empty:
                raise ValueError("Set cross_function_enabled=True in the cross-function audit and run calls first.")
            transfer_results_result = benchmark_coordinator.run_benchmarks(
                solver_type='cross_function',
                trials_per_condition=20,
                random_seed=42,
                first_instance_id=2,
                budget=100_000,
                trial_timeout_seconds=60.0,
                observation_trace_points=64,
                force_rerun=False,
                classical_baselines=['cmaes', 'de', 'pso'],
                noise_robustness_enabled=True,
                cross_function_enabled=False,
                cross_function_problem_ids=[1, 8, 11, 15, 21],
            )
            if transfer_results_result.is_success:
                transfer_results = transfer_results_result.value
            else:
                transfer_results = None
                print(f"Error: {transfer_results_result.error.message}")
        
        # Always export the current readiness matrix, without treating missing trials as failures.
        from shared.config import RESULTS_DIR
        transfer_report_dir = RESULTS_DIR / "reports"
        transfer_report_dir.mkdir(parents=True, exist_ok=True)
        workflow_result = benchmark_coordinator.audit_workload(
            solver_type='noise_robustness',
            trials_per_condition=20,
            random_seed=42,
            first_instance_id=2,
            budget=100_000,
            trial_timeout_seconds=60.0,
            observation_trace_points=64,
            force_rerun=False,
            classical_baselines=['cmaes', 'de', 'pso'],
            noise_robustness_enabled=True,
            cross_function_enabled=False,
            cross_function_problem_ids=[1, 8, 11, 15, 21],
        )
        if workflow_result.is_success:
            workflow_result.value.to_csv(
                transfer_report_dir / "noise_robustness_readiness_v1.csv", index=False
            )
            workflow_result = benchmark_coordinator.audit_workload(
                solver_type='cross_function',
                trials_per_condition=20,
                random_seed=42,
                first_instance_id=2,
                budget=100_000,
                trial_timeout_seconds=60.0,
                observation_trace_points=64,
                force_rerun=False,
                classical_baselines=['cmaes', 'de', 'pso'],
                noise_robustness_enabled=True,
                cross_function_enabled=False,
                cross_function_problem_ids=[1, 8, 11, 15, 21],
            )
            if workflow_result.is_success:
                workflow_result.value.to_csv(
                    transfer_report_dir / "cross_function_readiness_v1.csv", index=False
                )
            else:
                print(f"Error: {workflow_result.error.message}")
        else:
            print(f"Error: {workflow_result.error.message}")
    else:
        transfer_audit = None
        print(f"Error: {transfer_audit_result.error.message}")


Noise robustness: 540 conditions
Cross-function generalization: 0 conditions (opt-in)


,key,solver_type,solver,display_name,model,strategy,problem_id,dim,noise_std,mode,...,recorded_trials,reason,status,median_error,is_filtered,raw_key,source_problem_id,source_noise_std,code_hash,target_dir
0,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_ba...,noise_robustness,gwdg.qwen3-coder-next_baseline,gwdg.qwen3-coder-next (noise robustness),gwdg.qwen3-coder-next,baseline,1,2,0.05,explicit,...,0,missing_provenance,PENDING,None,False,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_ba...,1,0.0,6ed491585e56bc04d920fd8683cd696616242215861e3b...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
1,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_ba...,noise_robustness,gwdg.qwen3-coder-next_baseline,gwdg.qwen3-coder-next (noise robustness),gwdg.qwen3-coder-next,baseline,1,2,0.10,explicit,...,0,missing_provenance,PENDING,None,False,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_ba...,1,0.0,6ed491585e56bc04d920fd8683cd696616242215861e3b...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
2,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_ba...,noise_robustness,gwdg.qwen3-coder-next_baseline,gwdg.qwen3-coder-next (noise robustness),gwdg.qwen3-coder-next,baseline,1,2,0.20,explicit,...,0,missing_provenance,PENDING,None,False,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_ba...,1,0.0,6ed491585e56bc04d920fd8683cd696616242215861e3b...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
3,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_gu...,noise_robustness,gwdg.qwen3-coder-next_guided,gwdg.qwen3-coder-next (noise robustness),gwdg.qwen3-coder-next,guided,1,2,0.05,explicit,...,0,missing_provenance,PENDING,None,False,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_gu...,1,0.0,d0ec2ba3d15b0a36ae19debed92271e7d42a45eee63e97...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
4,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_gu...,noise_robustness,gwdg.qwen3-coder-next_guided,gwdg.qwen3-coder-next (noise robustness),gwdg.qwen3-coder-next,guided,1,2,0.10,explicit,...,0,missing_provenance,PENDING,None,False,gwdg.qwen3-coder-next/f1_2D_explicit_std0.0_gu...,1,0.0,d0ec2ba3d15b0a36ae19debed92271e7d42a45eee63e97...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen2.5-coder-32b-instruct-q4_k_m.gguf_thinking,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,21,50,0.10,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,3b32f93081c85d695a8437982f434f277a3cbdddb4a8dc...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
536,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen2.5-coder-32b-instruct-q4_k_m.gguf_thinking,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,21,50,0.20,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,3b32f93081c85d695a8437982f434f277a3cbdddb4a8dc...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
537,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen2.5-coder-32b-instruct-q4_k_m.gguf_vectori...,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,21,50,0.05,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,24a96a0b714823c6c20fd7de7a7e08c29c73119f18ff5a...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...
538,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,noise_robustness,qwen2.5-coder-32b-instruct-q4_k_m.gguf_vectori...,qwen2.5-coder-32b-instruct-q4_k_m.gguf (noise ...,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,21,50,0.10,explicit,...,0,missing_provenance,PENDING,None,False,qwen2.5-coder-32b-instruct-q4_k_m.gguf/f21_50D...,21,0.0,24a96a0b714823c6c20fd7de7a7e08c29c73119f18ff5a...,/Users/nicolaibrahim/Desktop/proj/AAD_LLM/resu...

""


## 5. Review outcomes

Failures remain in overall medians and target-success denominators. Finite-error summaries describe valid executions; review flags do not trigger synthesis reruns.


In [ ]:
evaluation_summary_result = benchmark_coordinator.summarize_results(
    trials_per_condition=20,
    random_seed=42,
    first_instance_id=2,
    budget=100_000,
    trial_timeout_seconds=60.0,
    observation_trace_points=64,
    force_rerun=False,
    classical_baselines=['cmaes', 'de', 'pso'],
    noise_robustness_enabled=True,
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    primary_target=primary_target,
    secondary_target=secondary_target,
)
evaluation_summary = evaluation_summary_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if evaluation_summary is not None:
    if evaluation_summary.empty:
        print("No current benchmark conditions to summarize.")
    else:
        print(f"Summary of {len(evaluation_summary)} conditions; primary target={primary_target:g}, secondary target={secondary_target:g}")
        display(pd.crosstab(evaluation_summary["Model"], evaluation_summary["Category"]))
        review_columns = ["Model", "Experiment ID", "Algorithm", "Mode", "Strategy", "Problem ID", "Dim", "Noise", "Executed trials", "Valid trials", "Expected trials", "Failed trials", "Execution rate", "Failure rate", "Solved trials", "Primary target rate", "Median error", "Median finite error", "Best baseline median error", "Category", "Suggested action"]
        failure_review = evaluation_summary[evaluation_summary["Category"].isin(["all_failed", "mostly_failed", "some_failed", "incomplete", "missing_or_stale"])][review_columns]
        large_error_review = evaluation_summary[evaluation_summary["Category"] == "large_error"][review_columns]
        formats = {"Execution rate": "{:.1%}", "Failure rate": "{:.1%}", "Primary target rate": "{:.1%}", "Noise": "{:g}", "Median error": "{:.3f}", "Median finite error": "{:.3f}", "Best baseline median error": "{:.3f}"}
        print(f"Failure/readiness review: {len(failure_review)} conditions (first 100 shown).")
        display(failure_review.head(100).style.format(formats, na_rep="—"))
        print(f"Large finite-error review: {len(large_error_review)} conditions (first 50 shown; compare with baselines before rerunning).")
        display(large_error_review.head(50).style.format(formats, na_rep="—"))
        print("Suggested experiment IDs to REVIEW for fresh synthesis, grouped by owning LLM:")
        for model, group in evaluation_summary[evaluation_summary["Rerun candidate"]].groupby("Model", sort=True):
            print(f"{model}: {sorted(group['Experiment ID'].dropna().astype(int).unique().tolist())}")
        print("No reruns launched. Review diagnostics first; explicitly select IDs in Notebook 02/configuration.")


Summary of 1696 conditions; primary target=1e-08, secondary target=0.01


Category,all_failed,finite_results,large_error,missing_or_stale,mostly_failed,some_failed
Model,,,,,,
cmaes,0,24,55,0,0,0
de,0,14,65,0,0,0
gwdg.qwen3-coder-next,0,164,326,1,1,7
pso,0,19,60,0,0,0
qwen2.5-coder-14b-instruct-q4_k_m.gguf,1,130,337,0,5,7
qwen2.5-coder-32b-instruct-q4_k_m.gguf,0,175,298,0,3,4


Failure/readiness review: 29 conditions (first 100 shown).


,Model,Experiment ID,Algorithm,Mode,Strategy,Problem ID,Dim,Noise,Executed trials,Valid trials,Expected trials,Failed trials,Execution rate,Failure rate,Solved trials,Primary target rate,Median error,Median finite error,Best baseline median error,Category,Suggested action
293,gwdg.qwen3-coder-next,6911,NoisyCMAES_Adapt,implicit,baseline,8,10,0,20,12,20,8,60.0%,40.0%,0,0.0%,2297.670,6.296,0.000,some_failed,Inspect failed-trial diagnostics
357,gwdg.qwen3-coder-next,7268,StableNoisyCMA,explicit,baseline,10,10,0.2,0,0,20,0,—,—,0,—,—,—,26.398,missing_or_stale,Refresh benchmark/code first
396,gwdg.qwen3-coder-next,7005,AdaptiveCMA_DE,explicit,vectorization,11,10,0,20,13,20,7,65.0%,35.0%,0,0.0%,10.698,4.839,0.000,some_failed,Inspect failed-trial diagnostics
399,gwdg.qwen3-coder-next,7015,RobustCovES,implicit,baseline,11,10,0.05,20,13,20,7,65.0%,35.0%,0,0.0%,8.538,6.937,0.241,some_failed,Inspect failed-trial diagnostics
402,gwdg.qwen3-coder-next,7012,NoisyAdaptiveCMA,explicit,thinking,11,10,0.05,20,10,20,10,50.0%,50.0%,0,0.0%,inf,101.654,0.241,some_failed,Inspect failed-trial diagnostics
404,gwdg.qwen3-coder-next,7013,ResampleES,explicit,vectorization,11,10,0.05,20,8,20,12,40.0%,60.0%,0,0.0%,inf,25.219,0.241,mostly_failed,Inspect; consider fresh synthesis
444,gwdg.qwen3-coder-next,7053,StochAdaptiveES,explicit,vectorization,11,50,0.1,20,18,20,2,90.0%,10.0%,0,0.0%,426.207,408.316,323.486,some_failed,Inspect failed-trial diagnostics
624,gwdg.qwen3-coder-next,7230,CovarAdaptDE,explicit,guided,21,50,0,20,19,20,1,95.0%,5.0%,0,0.0%,3.644,3.642,5.206,some_failed,Inspect failed-trial diagnostics
638,gwdg.qwen3-coder-next,7243,StochasticNES,explicit,baseline,21,50,0.1,20,15,20,5,75.0%,25.0%,0,0.0%,86.435,86.384,86.034,some_failed,Inspect failed-trial diagnostics
774,qwen2.5-coder-14b-instruct-q4_k_m.gguf,5533,ImprovedCMAESOptimizer,explicit,vectorization,1,10,0,20,8,20,12,40.0%,60.0%,0,0.0%,inf,27.952,0.000,mostly_failed,Inspect; consider fresh synthesis


Large finite-error review: 1141 conditions (first 50 shown; compare with baselines before rerunning).


,Model,Experiment ID,Algorithm,Mode,Strategy,Problem ID,Dim,Noise,Executed trials,Valid trials,Expected trials,Failed trials,Execution rate,Failure rate,Solved trials,Primary target rate,Median error,Median finite error,Best baseline median error,Category,Suggested action
12,cmaes,—,cmaes,explicit,classical,2,10,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,215.917,215.917,0.366,large_error,Compare with baselines; not an automatic rerun
13,cmaes,—,cmaes,explicit,classical,3,10,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,17.503,17.503,17.503,large_error,Compare with baselines; not an automatic rerun
14,cmaes,—,cmaes,explicit,classical,4,10,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,45.145,45.145,45.145,large_error,Compare with baselines; not an automatic rerun
16,cmaes,—,cmaes,explicit,classical,6,10,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,1.425,1.425,0.606,large_error,Compare with baselines; not an automatic rerun
17,cmaes,—,cmaes,explicit,classical,7,10,0.2,20,20,20,0,100.0%,0.0%,2,10.0%,1.387,1.387,1.325,large_error,Compare with baselines; not an automatic rerun
21,cmaes,—,cmaes,explicit,classical,8,2,0.2,20,20,20,0,100.0%,0.0%,7,35.0%,0.222,0.222,0.000,large_error,Compare with baselines; not an automatic rerun
23,cmaes,—,cmaes,explicit,classical,8,10,0.05,20,20,20,0,100.0%,0.0%,0,0.0%,8.782,8.782,8.243,large_error,Compare with baselines; not an automatic rerun
24,cmaes,—,cmaes,explicit,classical,8,10,0.1,20,20,20,0,100.0%,0.0%,0,0.0%,9.507,9.507,8.378,large_error,Compare with baselines; not an automatic rerun
25,cmaes,—,cmaes,explicit,classical,8,10,0.2,20,20,20,0,100.0%,0.0%,0,0.0%,9.971,9.971,9.971,large_error,Compare with baselines; not an automatic rerun
27,cmaes,—,cmaes,explicit,classical,8,50,0.05,20,20,20,0,100.0%,0.0%,0,0.0%,53.155,53.155,53.155,large_error,Compare with baselines; not an automatic rerun


Suggested experiment IDs to REVIEW for fresh synthesis, grouped by owning LLM:
gwdg.qwen3-coder-next: [7013]
qwen2.5-coder-14b-instruct-q4_k_m.gguf: [5379, 5479, 5494, 5533, 5579, 5672]
qwen2.5-coder-32b-instruct-q4_k_m.gguf: [5857, 5865, 5913]
No reruns launched. Review diagnostics first; explicitly select IDs in Notebook 02/configuration.
